In [1]:
import pandas as pd

In [2]:
FOLDER = "dados_implementacao/"
HUB = "hub_implementacao.csv"
TRIBUNAL = "tribunal_implementacao.csv"
APRESENTACOES = "apresentacoes_implementacao.json"
PAGAMENTOS = "pagamentos_implementacao.json"

In [3]:
df_hub = pd.read_csv(FOLDER + HUB)
df_tribunal = pd.read_csv(FOLDER + TRIBUNAL, encoding="ISO-8859-1", sep=";")
df_apresentacoes = pd.read_json(FOLDER + APRESENTACOES)
df_pagamentos = pd.read_json(FOLDER + PAGAMENTOS)

In [4]:
#Ajutando campo processo
df_tribunal.rename(columns={"numero_processo": "processo"}, inplace=True)
df_tribunal.insert(0, "numero_processo", 
                   df_tribunal["processo"].astype(str).str.replace(r"\D", "", regex=True))
df_tribunal = df_tribunal.drop(columns=["processo"])

#Ajustando campo valor_face
df_tribunal.insert(2, "valor_a_pagar", (
    df_tribunal["valor_face"].astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False).astype(float)
))
df_tribunal = df_tribunal.drop(columns=["valor_face"])

df_tribunal.head()

,numero_processo,ente_devedor,valor_a_pagar
0,00123451020198260053,Município de São Paulo,45000.0
1,00234562120208260053,Município de São Paulo,30000.0
2,00345673220188260405,Estado de São Paulo,12500.0
3,00456784320215020011,União,9800.0
4,00678906520168260053,Município de São Paulo,7000.0


In [5]:
#Ajustando campo numero_processo
df_hub.insert(0, "numero_processo", df_hub["processo"].astype(str).str.replace(r"\D", "", regex=True))
df_hub = df_hub.drop(columns=["processo"])

#Ajustando campo valor_face
df_hub.insert(2, "valor_a_pagar", (
    df_hub["valor_face"].astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False).astype(float)
))
df_hub = df_hub.drop(columns=["valor_face"])

#Ajustando data para o horário de brasília
datas_brasilia = pd.to_datetime(df_hub["apresentado_em"], format="ISO8601", utc=True)
datas_brasilia = datas_brasilia.dt.tz_convert("America/Sao_Paulo")

ate_2_abril = (datas_brasilia.dt.month < 4) | (datas_brasilia.dt.month == 4) & (datas_brasilia.dt.day <= 2)
ano_prazo = (datas_brasilia.dt.year + 1).where(ate_2_abril, datas_brasilia.dt.year + 2)

df_hub["prazo_pagamento"] = ano_prazo.astype(str) + "-12-31"

df_hub.head()

,numero_processo,ente,valor_a_pagar,apresentado_em,prazo_pagamento
0,00123451020198260053,MUNICIPIO DE SAO PAULO,450000.0,2025-04-02,2026-12-31
1,00234562120208260053,Município de São Paulo,310000.0,2025-04-03,2026-12-31
2,00345673220188260405,Estado de São Paulo,125000.0,2024-02-29,2025-12-31
3,00456784320215020011,Município de Santos,98000.0,2025-04-02T23:30:00-03:00,2026-12-31
4,00567895420174036100,Estado de São Paulo,220000.0,2025-04-03T01:00:00Z,2026-12-31


In [6]:
#Ajustando campo numero_processo
df_apresentacoes.insert(0, "numero_processo", df_apresentacoes["processo"].astype(str).str.replace(r"\D", "", regex=True))
df_apresentacoes = df_apresentacoes.drop(columns=["processo"])

df_apresentacoes.head()

,numero_processo,apresentado_em
0,00123451020198260053,2025-04-02
1,00234562120208260053,2025-04-03
2,00345673220188260405,2024-02-29
3,00456784320215020011,2025-04-02T23:30:00-03:00
4,00567895420174036100,2025-04-03T01:00:00Z


In [7]:
#Ajustando campo numero_processo
df_pagamentos.insert(0, "numero_processo", df_pagamentos["processo"].astype(str).str.replace(r"\D", "", regex=True))
df_pagamentos = df_pagamentos.drop(columns=["processo"])

#Ajustando campo valor
df_pagamentos.insert(2, "valor_pago", pd.to_numeric(
   df_pagamentos["valor"].astype(str).str.replace(r"\.(?=\d{3})", "", regex=True).str.replace(",", ".", regex=False), errors="coerce"
))
df_pagamentos = df_pagamentos.drop(columns=["valor"])

#Aplicando filtro, removendo duplicatas e ordenando campos
df_pagamentos = df_pagamentos.sort_values(by="atualizado_em")
df_pagamentos = df_pagamentos.drop_duplicates(subset=["evento_id"], keep="last")
df_pagamentos = df_pagamentos[df_pagamentos["valor_pago"].notna()]

df_pagamentos.head()

,numero_processo,evento_id,valor_pago,data_pagamento,tipo,atualizado_em
6,00345673220188260405,ev-006,5000.0,2025-02-20,pagamento,2025-02-20T10:00:00-03:00
7,00345673220188260405,ev-007,5000.0,2025-02-21,estorno,2025-02-21T10:00:00-03:00
8,00123451020198260053,ev-001,15500.0,2025-03-10,pagamento,2025-03-11T09:00:00-03:00
3,00123451020198260053,ev-002,2500.5,10/04/2025,pagamento,2025-04-10T09:00:00-03:00
2,00234562120208260053,ev-003,8000.0,2025-05-02,pagamento,2025-05-02T11:00:00-03:00


In [ ]:
#Criando a base
#Fazendo um LEFT JOIN do pagamentos_implementacao.json com apresentacoes_implementacao.json e tribunal.csv
df_base = (
    df_pagamentos.merge(
        df_hub[["numero_processo", "apresentado_em", "prazo_pagamento"]],
        on="numero_processo", how="left")
        
        .merge(
            df_tribunal[["numero_processo", "ente_devedor", "valor_a_pagar"]],
            on="numero_processo", how="left")
)

#Criando campo valor restante
df_base["valor_restante"] = df_base["valor_a_pagar"] - df_base["valor_pago"]
df_base.head()

,numero_processo,evento_id,valor_pago,data_pagamento,tipo,atualizado_em,apresentado_em,prazo_pagamento,ente_devedor,valor_a_pagar
0,00345673220188260405,ev-006,5000.0,2025-02-20,pagamento,2025-02-20T10:00:00-03:00,2024-02-29,2025-12-31,Estado de São Paulo,12500.0
1,00345673220188260405,ev-007,5000.0,2025-02-21,estorno,2025-02-21T10:00:00-03:00,2024-02-29,2025-12-31,Estado de São Paulo,12500.0
2,00123451020198260053,ev-001,15500.0,2025-03-10,pagamento,2025-03-11T09:00:00-03:00,2025-04-02,2026-12-31,Município de São Paulo,45000.0
3,00123451020198260053,ev-002,2500.5,10/04/2025,pagamento,2025-04-10T09:00:00-03:00,2025-04-02,2026-12-31,Município de São Paulo,45000.0
4,00234562120208260053,ev-003,8000.0,2025-05-02,pagamento,2025-05-02T11:00:00-03:00,2025-04-03,2026-12-31,Município de São Paulo,30000.0
